# Deploy an online endpoint whose scoring script lives *inside* the model asset

This notebook shows the **BYOC (bring-your-own-container)** pattern where the scoring script is packaged **inside the registered model asset** instead of being uploaded as a separate `code_configuration` code asset.

**Why BYOC?** In a standard managed deployment, `code_configuration.code` is always uploaded as a *separate* code asset - it cannot point inside the model. The model is mounted read-only at the `AZUREML_MODEL_DIR` environment variable. To serve a script that ships *with* the model, we use a custom container that starts the AzureML inference server (`azmlinfsrv`) and points `--entry_script` at the script inside `AZUREML_MODEL_DIR`.

**Approach in this notebook:**
1. Register the whole `model-1/` folder (the model **and** `onlinescoring/score_original.py`) as a single model asset.
2. Build a custom container from the AzureML minimal inference image that launches `azmlinfsrv` against the in-model script.
3. Create the deployment with **no** `code_configuration`.

> Prereq: a `.env` file in this folder with `SUBSCRIPTION_ID`, `DEV_RESOURCE_GROUP`, `DEV_WORKSPACE_NAME` (same pattern as the CPU notebook).

In [1]:
%pip install -r ../../../requirements.txt
print('Installation command executed. Restart the kernel if packages were updated.')

Note: you may need to restart the kernel to use updated packages.
Installation command executed. Restart the kernel if packages were updated.


c:\Users\jomedin\Documents\MLOPs-AzureML\.venv\Scripts\python.exe: No module named pip


## 1. Import libraries

In [2]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
    BuildContext,
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import os
import datetime

## 2. Connect to the Azure ML workspace

Loads workspace details from a local `.env` file (dev workspace), the same pattern used by the other notebooks in this folder.

In [3]:
from pathlib import Path

env_path = Path('.env')
subscription_id = ''
resource_group = ''
workspace = ''

if env_path.exists():
    with env_path.open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                k, v = line.split('=', 1)
                # strip surrounding double/single quotes without embedding quote chars
                v = v.strip().strip(chr(34)).strip(chr(39))
                os.environ[k.strip()] = v
    subscription_id = os.environ.get('SUBSCRIPTION_ID', '')
    resource_group = os.environ.get('DEV_RESOURCE_GROUP', '')
    workspace = os.environ.get('DEV_WORKSPACE_NAME', '')
else:
    print('No .env found - set subscription_id, resource_group, workspace manually below.')

print('subscription_id set:', bool(subscription_id))
print('resource_group:', resource_group)
print('workspace:', workspace)

subscription_id set: True
resource_group: rg-aml-ws-dev-cc-01
workspace: mlwdevcc01


In [4]:
ml_client = MLClient(DefaultAzureCredential(), subscription_id, resource_group, workspace)
print('Connected to workspace:', ml_client.workspace_name)

Connected to workspace: mlwdevcc01


## 3. Inspect the model folder

Notice the folder contains both the model (`model/sklearn_regression_model.pkl`) and the scoring script (`onlinescoring/score_original.py`). We register this **entire** folder as one model asset.

In [5]:
for root, dirs, files in os.walk('model-1'):
    for name in files:
        print(os.path.join(root, name))

model-1\sample-request.json
model-1\environment\conda-managedidentity.yml
model-1\environment\conda.yaml
model-1\model\sklearn_regression_model.pkl
model-1\onlinescoring\score.py
model-1\onlinescoring\score_managedidentity.py
model-1\onlinescoring\score_original.py


## 4. Register the whole folder as a single model asset

Registering `model-1/` (not just the `.pkl`) means the scoring script becomes part of the model asset and travels with it wherever the model is used.

In [6]:
model = Model(
    path='model-1',
    type=AssetTypes.CUSTOM_MODEL,
    name='taxi-model-embedded-scorer',
    description='Model asset that also contains its scoring script under onlinescoring/',
)
registered_model = ml_client.models.create_or_update(model)
print('Registered model:', registered_model.name, 'version', registered_model.version)

Uploading model-1 (0.01 MBs): 100%|##########| 8882/8882 [00:00<00:00, 18402.02it/s]




Registered model: taxi-model-embedded-scorer version 1


## 5. Build the custom container

The `Dockerfile` uses the current Python 3.11 slim image on Debian Trixie and installs `azureml-inference-server-http` explicitly. Python 3.11 retains wheel compatibility with the model's `scikit-learn==1.2.2` runtime. NumPy is pinned to 1.26.4 to preserve binary compatibility with that scikit-learn wheel.

The container starts `azmlinfsrv` with `--entry_script` pointing at the script **inside** the mounted model (`$AZUREML_MODEL_DIR`). `--model_dir` is set so the existing `score_original.py` finds the `.pkl` through `os.getenv('AZUREML_MODEL_DIR')`. The `if/else` covers both possible mount layouts (folder name preserved vs flattened).

In [7]:
os.makedirs('byoc_model_embedded', exist_ok=True)
print('Build context folder ready: byoc_model_embedded/')

Build context folder ready: byoc_model_embedded/


In [18]:
%%writefile byoc_model_embedded/Dockerfile
FROM python:3.11-slim-trixie

# Install the scoring script's native and Python runtime dependencies
RUN apt-get update \
    && apt-get install -y --no-install-recommends libgomp1 \
    && rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir --upgrade \
    pip \
    setuptools \
    "wheel>=0.46.2" \
    "jaraco.context>=6.1.0"

RUN pip install --no-cache-dir \
    azureml-inference-server-http==1.5.1 \
    scikit-learn==1.2.2 \
    joblib==1.5.2 \
    numpy==1.26.4 \
    pandas

# Clear the base image entrypoint so our CMD runs directly
ENTRYPOINT []

# Start the AzureML inference server against the scoring script that lives INSIDE
# the mounted model asset ($AZUREML_MODEL_DIR). The if/else handles both mount
# layouts: folder name preserved (model-1/...) vs flattened (...).
CMD ["sh", "-c", "if [ -f \"$AZUREML_MODEL_DIR/model-1/onlinescoring/score_original.py\" ]; then exec azmlinfsrv --entry_script \"$AZUREML_MODEL_DIR/model-1/onlinescoring/score_original.py\" --model_dir \"$AZUREML_MODEL_DIR/model-1/model\" --port 5001; else exec azmlinfsrv --entry_script \"$AZUREML_MODEL_DIR/onlinescoring/score_original.py\" --model_dir \"$AZUREML_MODEL_DIR/model\" --port 5001; fi"]

Overwriting byoc_model_embedded/Dockerfile


## 6. Define the BYOC environment

Providing `inference_config` (liveness / readiness / scoring routes) is what makes this a custom-container deployment. The routes match `azmlinfsrv` defaults (port 5001, `/` for liveness/readiness and `/score` for scoring).

In [14]:
env = Environment(
    name='byoc-inmodel-scorer',
    description='Custom container that serves a scoring script packaged inside the model asset',
    build=BuildContext(path='byoc_model_embedded'),
    inference_config={
        'liveness_route': {'port': 5001, 'path': '/'},
        'readiness_route': {'port': 5001, 'path': '/'},
        'scoring_route': {'port': 5001, 'path': '/score'},
    },
)

## 7. Create the endpoint

In [10]:
online_endpoint_name = 'endpoint-embed' + datetime.datetime.now().strftime('%m%d%H%M%f')
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description='BYOC endpoint whose scoring script is packaged inside the model asset',
    auth_mode='key',
)
ml_client.begin_create_or_update(endpoint).result()
print('Endpoint created:', online_endpoint_name)

Endpoint created: endpoint-embed07220921090687


## 8. Create the deployment (no `code_configuration`)

The scoring script is served from inside the model mount, so we omit `code_configuration` entirely. The first deployment can take ~10-15 minutes while the image is built in ACR.

In [20]:
registered_model = ml_client.models.get(
    name='taxi-model-embedded-scorer',
    label='latest',
)

existing_deployments = {
    item.name: item
    for item in ml_client.online_deployments.list(
        endpoint_name=online_endpoint_name,
    )
}
existing_blue = existing_deployments.get('blue')
if existing_blue and existing_blue.provisioning_state.lower() == 'failed':
    print('Deleting unrecoverable failed deployment: blue')
    ml_client.online_deployments.begin_delete(
        name='blue',
        endpoint_name=online_endpoint_name,
    ).result()

# Registration mutates env.version; clear it so a changed build gets a new version.
env.version = None
registered_environment = ml_client.environments.create_or_update(env)
print(
    'Using environment:',
    registered_environment.name,
    'version',
    registered_environment.version,
)

deployment = ManagedOnlineDeployment(
    name='blue',
    endpoint_name=online_endpoint_name,
    model=registered_model,   # the scoring script travels inside this asset
    environment=registered_environment,
    # No code_configuration: the script is served from inside the model mount
    instance_type='Standard_DS3_v2',
    instance_count=1,
)
ml_client.begin_create_or_update(deployment).result()

endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)
endpoint.traffic = {'blue': 100}
ml_client.begin_create_or_update(endpoint).result()
print('Deployment blue is live with 100% traffic')

Check: endpoint endpoint-embed07220921090687 exists


Using environment: byoc-inmodel-scorer version 3
....................................................................

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


Deployment blue is live with 100% traffic


## 9. Test the endpoint

In [24]:
import time
from azure.core.exceptions import ServiceResponseError

for attempt in range(1, 6):
    try:
        response = ml_client.online_endpoints.invoke(
            endpoint_name=online_endpoint_name,
            deployment_name='blue',
            request_file='model-1/sample-request.json',
        )
        break
    except ServiceResponseError:
        if attempt == 5:
            raise
        print(f'Endpoint connection reset during rollout; retrying ({attempt}/5)...')
        time.sleep(15)

print(response)

[11055.977245525679, 4503.079536107787, 11055.977245525679, 4503.079536107787]


## 10. Deployment logs & troubleshooting

If you see a `FileNotFoundError` for the model or entry script, list `AZUREML_MODEL_DIR` in the logs to confirm the mount layout, then adjust the paths in the `Dockerfile` `CMD` (the `if/else` already covers the two common layouts).

In [23]:
print(ml_client.online_deployments.get_logs(name='blue', endpoint_name=online_endpoint_name, lines=100))

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Killing, Type: Normal, Time: 2026-07-22T14:27:18.787916Z, Message: Stopping container inference-server
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-07-22T14:27:29.899306Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-07-22T14:27:29.903271Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-07-22T14:28:00.518161Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-07-22T14:28:00.518161Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-07-22T14:28:00.568333Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-07-22T14:28:00.633072Z, Message: Started container inference-server
Kind: Pod, Name: Contai

## 11. (Optional) Test the same wiring locally

You can validate the in-model script on your machine before deploying (requires `azureml-inference-server-http` installed locally):

```bash
cd notebooks/deployments/online/custom_scoring_script/model-1
azmlinfsrv --entry_script ./onlinescoring/score_original.py --model_dir ./model
# in another terminal:
curl -X POST http://127.0.0.1:5001/score -H "Content-Type: application/json" --data @sample-request.json
```

## 12. Clean up

Delete the endpoint (and its deployment) when you are done to stop incurring cost.

In [ ]:
ml_client.online_endpoints.begin_delete(name=online_endpoint_name)